In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Voting

In [2]:
from sklearn.ensemble import VotingClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [3]:
X, y = make_classification(
    n_samples = 500,
    n_features = 20,
    n_informative = 5,
    n_redundant = 2,
    random_state = 42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.3, random_state = 42
)

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [5]:
lr = LogisticRegression()
svc = SVC()
dtc = DecisionTreeClassifier(max_depth=3)

In [6]:
voting_clf = VotingClassifier(
    estimators = [
        ("lr",lr),
        ("svc",svc),
        ("dtc",dtc)
    ]
)

In [7]:
#hard voting this one which uses mode and mean of predictions while soft voting uses probabilities like the naive bayes

In [8]:
voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('lr', LogisticRegression()), ('svc', SVC()),
                             ('dtc', DecisionTreeClassifier(max_depth=3))])

In [9]:
y_pred = voting_clf.predict(X_test)

print("accuracy-",accuracy_score(y_pred, y_test))
print("cr-",classification_report(y_pred, y_test))

accuracy- 0.8333333333333334
cr-               precision    recall  f1-score   support

           0       0.86      0.83      0.85        83
           1       0.80      0.84      0.82        67

    accuracy                           0.83       150
   macro avg       0.83      0.83      0.83       150
weighted avg       0.83      0.83      0.83       150



In [10]:
from sklearn.datasets import make_regression
X, y = make_regression(
    n_samples = 500,
    n_features = 20,
    n_informative = 5,
    random_state = 42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.3, random_state = 42
)

In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

In [12]:
lin_reg = LinearRegression()
dtr = DecisionTreeRegressor()
svr = SVR()

In [13]:
from sklearn.ensemble import VotingRegressor

In [14]:
vr = VotingRegressor(
    estimators = [
        ("lr", lin_reg),
        ("dtr", dtr),
        ("svr", svr)
    ]
)

In [15]:
vr.fit(X_train, y_train)

VotingRegressor(estimators=[('lr', LinearRegression()),
                            ('dtr', DecisionTreeRegressor()), ('svr', SVR())])

In [16]:
y_pred = vr.predict(X_test)

In [17]:
from sklearn.metrics import r2_score

print("r^2", r2_score(y_test, y_pred))

r^2 0.8408539460686645


# Stacking

In [18]:
# classifier
from sklearn.ensemble import StackingClassifier

In [19]:
meta_model = LogisticRegression()

stacking_clf = StackingClassifier(
    estimators = [
        ("lr",lr),
        ("svc",svc),
        ("dtc",dtc)
    ],
    final_estimator = meta_model,
cv=5
)

In [20]:
X, y = make_classification(
    n_samples = 500,
    n_features = 20,
    n_informative = 5,
    n_redundant = 2,
    random_state = 42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.3, random_state = 42
)

In [21]:
stacking_clf.fit(X_train, y_train)

StackingClassifier(cv=5,
                   estimators=[('lr', LogisticRegression()), ('svc', SVC()),
                               ('dtc', DecisionTreeClassifier(max_depth=3))],
                   final_estimator=LogisticRegression())

In [22]:
y_pred = stacking_clf.predict(X_test)
print("accuracy", accuracy_score(y_test, y_pred))

accuracy 0.88


In [23]:
# regressor
from sklearn.ensemble import StackingRegressor

In [24]:
sr = StackingRegressor(
    estimators = [
        ("lr", lin_reg),
        ("dtr", dtr),
        ("svr", svr)
    ],
    cv=5
)
sr.fit(X_train, y_train)

StackingRegressor(cv=5,
                  estimators=[('lr', LinearRegression()),
                              ('dtr', DecisionTreeRegressor()),
                              ('svr', SVR())])

In [25]:
y_pred = sr.predict(X_test)
y_pred_train = sr.predict(X_train)

print("r2 test score", r2_score(y_test, y_pred))
print("r2 train score", r2_score(y_train, y_pred_train))

r2 test score 0.6225263035464219
r2 train score 0.9425700825175789
